[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LinkedInLearning/Fine-Tuning-for-LLMs-from-Beginner-to-Advanced/blob/main/CH04/04_05_end/notebooks/04_05_validate_and_split.ipynb)

# 04_05 Split and validate your dataset

Check every synthetic example before it reaches training, then split the survivors into training and evaluation sets.

These steps don't need a GPU. The default CPU runtime works fine.

In [ ]:
# Pull the course files into this Colab session and load the Ridgeline helpers.
!git clone --quiet https://github.com/LinkedInLearning/Fine-Tuning-for-LLMs-from-Beginner-to-Advanced.git course 2>/dev/null || echo "Course files already downloaded."

import sys
sys.path.insert(0, "course/CH04")

import ridgeline

In [ ]:
raw = ridgeline.load_tickets(ridgeline.DATA / "synthetic_raw.jsonl")
seeds = ridgeline.load_tickets()
print(f"{len(raw)} synthetic tickets, {len(seeds)} hand-written seed tickets")

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("unsloth/Qwen3-4B-Instruct-2507")

## Define what makes an example unusable

An example gets dropped if it's missing a field, uses an unknown label, breaks a voice rule, or looks like it contains personal information.

In [ ]:
import re

REQUIRED = ["customer_name", "subject", "message", "context", "category", "escalate", "reply"]
PII_PATTERNS = {
    "email address": r"[\w.+-]+@[\w-]+\.[\w.]+",
    "phone number": r"\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b",
    "card number": r"\b(?:\d[ -]?){13,16}\b",
}


def find_problems(ticket):
    """Return a list of reasons this ticket can't be used for training."""
    missing = [field for field in REQUIRED if not ticket.get(field)]
    if missing:
        return [f"missing {', '.join(missing)}"]

    problems = []
    if ticket["category"] not in ridgeline.CATEGORIES:
        problems.append("unknown category")
    if ticket["escalate"] not in ("Yes", "No"):
        problems.append("escalate is not Yes or No")
    problems += ridgeline.check_reply(ticket, ridgeline.gold_output(ticket))["failures"]

    text = " ".join(ticket[field] for field in REQUIRED)
    for name, pattern in PII_PATTERNS.items():
        if re.search(pattern, text):
            problems.append(f"possible {name}")
    return problems

## Check every ticket

In [ ]:
from collections import Counter

kept, reasons = [], Counter()
for ticket in raw:
    problems = find_problems(ticket)
    if problems:
        reasons.update({p.split(" (")[0].split(" '")[0].split(":")[0] for p in problems})
    else:
        kept.append(ticket)

print(f"Kept {len(kept)} of {len(raw)} tickets")
for reason, count in reasons.most_common():
    print(f"  {count:>3}  {reason}")

## Remove duplicates

Generators repeat themselves. Two copies of the same ticket teach the model that pattern twice as hard.

In [ ]:
seen, unique = set(), []
for ticket in kept:
    key = " ".join(ticket["message"].lower().split())
    if key not in seen:
        seen.add(key)
        unique.append(ticket)

print(f"Removed {len(kept) - len(unique)} duplicates, {len(unique)} left")

## Check lengths and balance

Every example has to fit in 2,048 tokens. Then look at how many examples each category has left.

In [ ]:
MAX_TOKENS = 2048

examples = [ridgeline.to_training_example(t) for t in unique]
examples = [ex for ex in examples
            if ridgeline.count_tokens(tokenizer, ex["messages"]) <= MAX_TOKENS]
print(f"{len(examples)} examples fit in {MAX_TOKENS} tokens")

print("\nExamples per category:")
for category, count in sorted(Counter(ex["category"] for ex in examples).items()):
    print(f"  {count:>3}  {category}")

## Split into training and evaluation sets

The 18 hand-written seed tickets go into evaluation, along with five synthetic tickets per category. The model never trains on any of them, so they show how it handles tickets it hasn't seen.

In [ ]:
import random

rng = random.Random(42)
EVAL_PER_CATEGORY = 5

eval_set = [ridgeline.to_training_example(t) for t in seeds]
train_set = []
for category in ridgeline.CATEGORIES:
    group = [ex for ex in examples if ex["category"] == category]
    rng.shuffle(group)
    eval_set += group[:EVAL_PER_CATEGORY]
    train_set += group[EVAL_PER_CATEGORY:]
rng.shuffle(train_set)

print(f"Training examples: {len(train_set)}")
print(f"Evaluation examples: {len(eval_set)} (18 seeds plus {EVAL_PER_CATEGORY} per category)")

## Save the files

In [ ]:
ridgeline.save_jsonl(train_set, "train.jsonl")
ridgeline.save_jsonl(eval_set, "eval.jsonl")
print("Saved train.jsonl and eval.jsonl. Download them from the Files panel to keep them.")